# Урок 10 - AI Агенти у продукції

У цьому уроці ви дізнаєтесь про **продуктивні патерни** для AI агентів за допомогою Microsoft Agent Framework (Python). Ми розглянемо:

- **Спостереження** — додавання таймінгу та логування взаємодій агента
- **Оцінювання** — використання агента-оцінювача для оцінки якості відповідей
- **Управління витратами** — стратегії оптимізації токенів та вибору моделей

Сценарій — **туристичний агент**, який допомагає користувачам планувати подорожі, з додатковим моніторингом та оцінюванням.


## Налаштування


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Виробничі міркування

Перехід агентів штучного інтелекту від прототипів до виробництва вимагає ретельної уваги до трьох основ:

1. **Спостережуваність** — Потрібно мати видимість того, що робить агент, скільки часу це займає та які інструменти він викликає. Без трасування та логування налагодження проблем у виробництві майже неможливе.

2. **Оцінка** — Автоматизовані перевірки якості гарантують, що відповіді агента залишаються точними, повними та корисними з часом. Оцінювальний агент може оцінювати відповіді відповідно до визначених критеріїв.

3. **Управління витратами** — Використання токенів безпосередньо впливає на вартість. Стратегії, такі як оптимізація запитів, вибір моделі та кешування, допомагають контролювати витрати без шкоди для якості.


## Створення Observable Агента

Ми визначаємо інструменти подорожі та обгортаємо виклик агента з таймінгом, щоб ми могли моніторити затримку. У продакшені ви б інтегрували це з OpenTelemetry або подібною системою трасування.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Evaluation Patterns

A common production pattern is to use a second agent as an **evaluator**. The evaluator scores the primary agent's response against predefined criteria such as completeness, accuracy, and helpfulness.

This enables:
- Automated quality gates before responses reach users
- Regression detection when prompts or models change
- Continuous monitoring of agent performance over time


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Стратегії управління витратами

Контроль витрат має вирішальне значення для AI агентів виробництва. Ось ключові стратегії:

| Стратегія | Опис |
|---|---|
| **Оптимізація запитів** | Тримайте інструкції системи короткими. Видаляйте зайвий контекст, щоб зменшити кількість вхідних токенів. |
    "| **Вибір моделі** | Використовуйте менші, дешевші моделі (наприклад, GPT-5-mini) для простих завдань, таких як класифікація чи вилучення, і залишайте більші моделі для складного мислення. |\n",
| **Кешування** | Кешуйте результати інструментів та часті запити, щоб уникнути зайвих API викликів. |
| **Бюджети токенів** | Встановлюйте обмеження `max_tokens`, щоб запобігти несподівано довгим відповідям. |
| **Пакетування** | Групуйте кілька запитів користувачів в один виклик API, якщо це можливо. |

На практиці добре працює поетапний підхід: спрямовуйте прості запити до швидкої, недорогої моделі і передавайте лише складніші запити більш потужній.


## Резюме

У цьому уроці ви дізналися, як:

1. **Додати спостережуваність** до взаємодій агентів за допомогою вимірювання часу та логування, створюючи основу для трасування та моніторингу.
2. **Автоматично оцінювати відповіді агентів**, використовуючи агент-оцінювач, який оцінює повноту, точність і корисність.
3. **Керувати витратами** через оптимізацію запитів, вибір моделей, кешування та бюджети токенів.

Ці виробничі шаблони допомагають забезпечити надійність, вимірюваність і економічність ваших агентів штучного інтелекту у масштабі.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
